# Cross-structure OFF period relationships

Analyze temporal relationships between OFF periods detected across brain structures
within a single multi-structure subject. Covers local vs global sleep, pairwise overlaps,
event-locked OFF onset probability, and OFF property correlates of overlap.


In [ ]:
import itertools
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pubplots as pp
import seaborn as sns
from scipy.stats import fisher_exact, spearmanr

from cnpix_local_sleep import const, hyp, plots
from cnpix_local_sleep.morphological.pipeline import cross_structure_offs as csx

## Parameters and configuration

OFFs are loaded from the aggregated `llas_offs.parquet` file (cortical spatial OFFs
only), following the pattern in `viz_bandpower_vs_off.ipynb`. LLAS/CLAS/BLAS categories
are assigned by set-membership against the more restrictive parquets.


In [ ]:
# --- OFF source & single-condition scope selectors ---
# OFF_SOURCE selects which OFF detection feeds every plot. One of:
#   "morphological-full48h" (default) -> in-memory full-48h morphological OFFs, subset
#                                     to the six statistical conditions
#   "morphological"                   -> per-condition aggregated parquets
OFF_SOURCE = "morphological-full48h"

# SINGLE_COND_SCOPE controls the plots that show a single slice of the data
# (PETHs, cross-correlograms, local-vs-overlapping, jitter control):
#   "whole_recording" (default) -> all OFFs across the 48h recording
#                                  (requires OFF_SOURCE="morphological-full48h")
#   "condition"                 -> only SINGLE_CONDITION's OFFs
SINGLE_COND_SCOPE = "whole_recording"
SINGLE_CONDITION = "Early.BSL.NREM"

subject = "CNPIX12-Santiago"
conditions = list(const.CORE_CONDITIONS)

assert OFF_SOURCE in csx.OFF_SOURCES, OFF_SOURCE
assert SINGLE_COND_SCOPE in ("whole_recording", "condition")
assert SINGLE_COND_SCOPE != "whole_recording" or OFF_SOURCE == "morphological-full48h", (
    "SINGLE_COND_SCOPE='whole_recording' requires OFF_SOURCE='morphological-full48h'"
)

NOTEBOOK_NAME = "cross_structure_off_relationships"
OUTPUT_DIR = Path(f"./outputs/{NOTEBOOK_NAME}/{OFF_SOURCE}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"OFF_SOURCE={OFF_SOURCE}  SINGLE_COND_SCOPE={SINGLE_COND_SCOPE}")
print(f"Saving SVGs to {OUTPUT_DIR}")

In [ ]:
# Load condition-subset, event-level OFFs across all of the subject's cortical
# structures, with an ordered LLAS/CLAS/BLAS `category` column. Routed by
# OFF_SOURCE (full-48h in-memory vs per-condition parquets).
offs = csx.load_cross_structure_offs(OFF_SOURCE, subject=subject, conditions=conditions)

structure_labels = sorted(offs["structure"].dropna().unique())
n_structures = len(structure_labels)

print(f"Subject: {subject}")
print(f"Structures ({n_structures}): {structure_labels}")
print(f"Conditions ({len(conditions)}): {conditions}")
print(f"\n{len(offs)} OFFs: {offs['category'].value_counts().to_dict()}")
print("\nPer-structure counts:")
print(offs.groupby("structure", observed=True).size().to_string())

## Load OFFs

In [ ]:
# Build offs_dict keyed by (structure, condition) for the per-condition plots.
offs_dict = csx._build_offs_dict(offs, structure_labels, conditions)

# Summary table
summary_rows = []
for (structure, cond), df in sorted(offs_dict.items()):
    summary_rows.append(
        {
            "structure": structure,
            "condition": cond,
            "n_offs": len(df),
            "mean_duration": df["duration"].mean() if len(df) > 0 else np.nan,
            "total_off_time": df["duration"].sum() if len(df) > 0 else 0.0,
        }
    )
summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

In [ ]:
# Load condition hypnograms for condition duration computation.
# Use the first probe found in the data as reference (all probes share the
# same time axis after synchronization).
ref_probe = offs["probe"].iloc[0]
hgs = hyp.load_statistical_condition_hypnograms(subject, ref_probe)

condition_durations = {}
for cond in conditions:
    condition_durations[cond] = hgs[cond]["duration"].sum()
    print(
        f"{cond}: {condition_durations[cond]:.1f}s "
        f"({condition_durations[cond] / 3600:.2f}h)"
    )

In [ ]:
# --- Resolve the single-condition scope used by the per-slice plots below ---
# Builds scope_* structures parallel to the per-condition ones but keyed by a
# single pseudo-condition (scope_cond). Plot filenames use scope_label.
if SINGLE_COND_SCOPE == "whole_recording":
    scope_offs = csx.load_whole_recording_offs(OFF_SOURCE, subject=subject)
    scope_cond = scope_offs["condition"].iloc[0]  # "Full48h"
    scope_label = "whole_recording"
else:
    scope_offs = offs[offs["condition"] == SINGLE_CONDITION].reset_index(drop=True)
    scope_cond = SINGLE_CONDITION
    scope_label = SINGLE_CONDITION

scope_offs_dict = csx._build_offs_dict(scope_offs, structure_labels, [scope_cond])
scope_overlap_counts = csx.compute_overlap_counts(
    scope_offs_dict, structure_labels, scope_cond
)
scope_pair_overlaps = {
    (sa, sb, scope_cond): csx.compute_pairwise_overlaps(
        scope_offs_dict[(sa, scope_cond)], scope_offs_dict[(sb, scope_cond)]
    )
    for sa, sb in itertools.combinations(structure_labels, 2)
}
if SINGLE_COND_SCOPE == "condition":
    scope_T = condition_durations.get(scope_cond, np.nan)
else:
    _starts = scope_offs["start_time"]
    scope_T = (_starts.max() - _starts.min()) if len(scope_offs) else np.nan

print(f"Scope: {scope_label}  ({len(scope_offs)} OFFs, scope_cond={scope_cond})")

## Local vs global sleep

For each OFF in each structure, count how many other structures have a temporally
overlapping OFF. An OFF is local if it occurs in only one structure (overlap count = 0),
and progressively more global as more structures overlap.


In [ ]:
# Reuse the shared, tested implementation from the pipeline module.
compute_overlap_counts = csx.compute_overlap_counts

In [ ]:
# Compute and display local/global fractions for each condition.
all_overlap_counts = {}  # (condition,) -> dict[struct -> array]

for cond in conditions:
    overlap_counts = compute_overlap_counts(offs_dict, structure_labels, cond)
    all_overlap_counts[cond] = overlap_counts

    print(f"\n=== {cond} ===")
    for struct in structure_labels:
        counts = overlap_counts[struct]
        if len(counts) == 0:
            print(f"  {struct}: no OFFs")
            continue
        n_local = np.sum(counts == 0)
        n_any_overlap = np.sum(counts > 0)
        frac_local = n_local / len(counts)
        print(
            f"  {struct}: {len(counts)} OFFs, "
            f"{n_local} local ({frac_local:.1%}), "
            f"{n_any_overlap} overlapping ({1 - frac_local:.1%}), "
            f"mean overlap with {np.mean(counts):.2f} other structures"
        )

In [ ]:
# Stacked bar plot of overlap degree by structure (conditions nested within structure).
with pp.destination("default"):
    fig, ax = plt.subplots(figsize=(14, 5))

    max_overlap = n_structures - 1
    _flare_cmap = sns.color_palette("flare", as_cmap=True)
    colors = _flare_cmap(np.linspace(0.1, 0.9, max_overlap + 1))

    n_conditions = len(conditions)
    group_width = 0.8
    bar_width = group_width / n_conditions
    offsets = np.linspace(
        -group_width / 2 + bar_width / 2,
        group_width / 2 - bar_width / 2,
        n_conditions,
    )
    x_positions = np.arange(n_structures)

    # Pre-compute fractions: fracs[struct_idx][cond_idx][degree]
    fracs = np.zeros((n_structures, n_conditions, max_overlap + 1))
    for si, struct in enumerate(structure_labels):
        for ci, cond in enumerate(conditions):
            counts = all_overlap_counts[cond][struct]
            for degree in range(max_overlap + 1):
                fracs[si, ci, degree] = (
                    np.mean(counts == degree) if len(counts) > 0 else 0.0
                )

    overlap_handles = []
    for degree in range(max_overlap + 1):
        label = "Local (0)" if degree == 0 else f"Overlap w/ {degree}"
        for si in range(n_structures):
            for ci in range(n_conditions):
                x = x_positions[si] + offsets[ci]
                bottom = fracs[si, ci, :degree].sum()
                bar = ax.bar(
                    x,
                    fracs[si, ci, degree],
                    width=bar_width * 0.9,
                    bottom=bottom,
                    color=colors[degree],
                    edgecolor="white",
                    linewidth=0.3,
                    label=label if (ci == 0 and si == 0) else "_nolegend_",
                )
        overlap_handles.append(plt.Rectangle((0, 0), 1, 1, color=colors[degree]))

    for si in range(n_structures):
        for ci, cond in enumerate(conditions):
            x = x_positions[si] + offsets[ci]
            ax.text(
                x, -0.04, cond, ha="center", va="top", fontsize=5, rotation=45,
                transform=ax.get_xaxis_transform(),
            )

    ax.set_xticks(x_positions)
    ax.set_xticklabels(structure_labels, fontsize=10)
    ax.tick_params(axis="x", which="both", bottom=False)
    ax.set_ylabel("Fraction of OFFs")
    ax.set_ylim(0, 1)

    overlap_legend_labels = [
        "Local (0)" if d == 0 else f"Overlap w/ {d}" for d in range(max_overlap + 1)
    ]
    ax.legend(
        overlap_handles, overlap_legend_labels, loc="upper right", fontsize=7,
        title="# overlapping structures",
    )
    fig.suptitle(f"{subject} | Local vs Global OFFs", fontsize=13)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "overlap_degree_bars.svg")

In [ ]:
# Chance baseline: compare observed local fraction to expected under
# independence (product of (1 - p_off) across other structures).
for cond in conditions:
    T = condition_durations[cond]
    overlap_counts = all_overlap_counts[cond]
    print(f"\n=== {cond}: Expected vs Observed Local Fraction ===")
    for ref_struct in structure_labels:
        counts = overlap_counts[ref_struct]
        if len(counts) == 0:
            continue
        # Under independence, P(no overlap) = product of (1 - p_off_other)
        p_independent_no_overlap = 1.0
        for other_struct in structure_labels:
            if other_struct == ref_struct:
                continue
            other_offs = offs_dict[(other_struct, cond)]
            p_off = other_offs["duration"].sum() / T if not other_offs.empty else 0
            p_independent_no_overlap *= 1 - p_off

        expected_local = p_independent_no_overlap
        observed_local = np.mean(counts == 0)

        print(
            f"  {ref_struct}: "
            f"observed local={observed_local:.1%}, "
            f"expected local (independence)={expected_local:.1%}, "
            f"ratio={observed_local / expected_local:.2f}"
        )

## Pairwise overlap distributions

For each pair of structures, characterize their OFF overlaps: fraction of overlap,
temporal onset offset, and number of overlapping against non-overlapping OFFs.


In [ ]:
compute_pairwise_overlaps = csx.compute_pairwise_overlaps

In [ ]:
# Compute pairwise overlaps for all structure pairs and conditions.
pair_overlaps = {}  # (struct_a, struct_b, condition) -> DataFrame

structure_pairs = list(itertools.combinations(structure_labels, 2))
print(f"Number of structure pairs: {len(structure_pairs)}")

for struct_a, struct_b in structure_pairs:
    for cond in conditions:
        overlaps = compute_pairwise_overlaps(
            offs_dict[(struct_a, cond)],
            offs_dict[(struct_b, cond)],
        )
        pair_overlaps[(struct_a, struct_b, cond)] = overlaps

        n_a = len(offs_dict[(struct_a, cond)])
        n_b = len(offs_dict[(struct_b, cond)])
        n_a_with = overlaps["index_a"].nunique() if not overlaps.empty else 0
        n_b_with = overlaps["index_b"].nunique() if not overlaps.empty else 0
        print(
            f"  {struct_a}-{struct_b} ({cond}): "
            f"{len(overlaps)} pairs, "
            f"{n_a_with}/{n_a} A overlap, "
            f"{n_b_with}/{n_b} B overlap"
        )

In [ ]:
def _get_pairwise_overlap_frac(
    pair_overlaps: dict,
    offs_dict: dict,
    structure_labels: list[str],
    cond: str,
) -> np.ndarray:
    """Build asymmetric matrix: mat[i,j] = fraction of structure i's OFFs
    that overlap with at least one OFF in structure j."""
    n = len(structure_labels)
    mat = np.full((n, n), np.nan)
    for i, sa in enumerate(structure_labels):
        n_a = len(offs_dict[(sa, cond)])
        mat[i, i] = 1.0
        for j, sb in enumerate(structure_labels):
            if i == j:
                continue
            # Try (sa, sb) then (sb, sa)
            key = (sa, sb, cond)
            key_rev = (sb, sa, cond)
            if key in pair_overlaps:
                ov = pair_overlaps[key]
                n_with = ov["index_a"].nunique() if not ov.empty else 0
            elif key_rev in pair_overlaps:
                ov = pair_overlaps[key_rev]
                n_with = ov["index_b"].nunique() if not ov.empty else 0
            else:
                n_with = 0
            mat[i, j] = n_with / n_a if n_a > 0 else 0
    return mat

In [ ]:
# Heatmap of pairwise overlap fractions.
with pp.destination("default"):
    n_cols = min(3, len(conditions))
    n_rows = int(np.ceil(len(conditions) / n_cols))
    fig, axes = plt.subplots(
        n_rows, n_cols, figsize=(5.5 * n_cols, 4.5 * n_rows), squeeze=False,
    )

    for idx, cond in enumerate(conditions):
        ax = axes[idx // n_cols, idx % n_cols]
        mat = _get_pairwise_overlap_frac(pair_overlaps, offs_dict, structure_labels, cond)
        sns.heatmap(
            mat, xticklabels=structure_labels, yticklabels=structure_labels,
            annot=True, fmt=".2f", cmap="YlOrRd", vmin=0, vmax=1, ax=ax,
            cbar_kws={"label": "Fraction overlapping"},
        )
        ax.set_title(cond, fontsize=9)
        ax.set_ylabel("Reference structure")
        ax.set_xlabel("Other structure")

    for idx in range(len(conditions), n_rows * n_cols):
        axes[idx // n_cols, idx % n_cols].set_visible(False)

    fig.suptitle(
        f"{subject} | Fraction of OFFs overlapping with another structure", fontsize=13,
    )
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "pairwise_overlap_heatmaps.svg")

In [ ]:
# Detail histograms for the chosen scope's selected pairs.
highlight_pairs = structure_pairs[:4]
cond = scope_cond
hist_color = "steelblue"

with pp.destination("default"):
    fig, axes = plt.subplots(
        len(highlight_pairs), 4, figsize=(22, 3 * len(highlight_pairs)), squeeze=False,
    )
    for row, (sa, sb) in enumerate(highlight_pairs):
        key = (sa, sb, cond) if (sa, sb, cond) in scope_pair_overlaps else (sb, sa, cond)
        ov = scope_pair_overlaps.get(key, pd.DataFrame())
        if ov.empty:
            for col in range(4):
                axes[row, col].text(
                    0.5, 0.5, "No overlaps", ha="center", va="center",
                    transform=axes[row, col].transAxes,
                )
            axes[row, 0].set_title(f"{sa} vs {sb}")
            continue

        median_overlap_dur = ov["overlap_duration"].median() * 1000
        axes[row, 0].hist(ov["overlap_duration"] * 1000, bins=50, color=hist_color, alpha=0.7)
        axes[row, 0].set_xlabel("Overlap duration (ms)")
        axes[row, 0].set_ylabel("Count")
        axes[row, 0].set_title(f"{sa} vs {sb}: overlap duration (median={median_overlap_dur:.1f}ms)")

        axes[row, 1].hist(ov["fraction_of_a"], bins=30, color=hist_color, alpha=0.7)
        axes[row, 1].set_xlabel(f"Overlap / duration({sa})")
        axes[row, 1].set_ylabel("Count")
        axes[row, 1].set_title(f"{sa} vs {sb}: overlap fraction")
        axes[row, 1].set_xlim(0, 1)

        median_onset_lag = ov["onset_lag"].median() * 1000
        axes[row, 2].hist(ov["onset_lag"] * 1000, bins=50, color=hist_color, alpha=0.7)
        axes[row, 2].axvline(0, color="k", ls="--", lw=0.8)
        axes[row, 2].set_xlabel(f"Onset lag {sb} - {sa} (ms)")
        axes[row, 2].set_ylabel("Count")
        axes[row, 2].set_title(f"{sa} vs {sb}: onset lag (median={median_onset_lag:.1f}ms)")

        median_offset_lag = ov["offset_lag"].median() * 1000
        axes[row, 3].hist(ov["offset_lag"] * 1000, bins=50, color=hist_color, alpha=0.7)
        axes[row, 3].axvline(0, color="k", ls="--", lw=0.8)
        axes[row, 3].set_xlabel(f"Offset lag {sb} - {sa} (ms)")
        axes[row, 3].set_ylabel("Count")
        axes[row, 3].set_title(f"{sa} vs {sb}: offset lag (median={median_offset_lag:.1f}ms)")

    fig.suptitle(f"{subject} | {cond} | Pairwise overlap details", fontsize=13)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"pairwise_overlap_details_{scope_label}.svg")

## OFF-onset-locked histograms (PETH-like)

Peri-event time histogram of OFF onsets in each target structure, time-locked to OFF
onsets in a reference structure. Analogous to a cross-correlogram, but for OFF events
rather than spikes.

The full pairwise cross-correlogram matrix below uses the convention that rows are the
reference structure (each row's OFF onsets define time zero) and columns are the target
structure (the events being counted). Reading across a row shows how all other
structures' onsets relate to that row's structure; reading down a column shows how that
structure's onsets relate to all reference structures. Diagonal entries are
auto-correlograms.


In [ ]:
compute_event_peth = csx.compute_event_peth

In [ ]:
# PETHs: all target structures locked to one reference structure.
reference_struct = structure_labels[0]
cond = scope_cond
window = 0.5  # +/- 500ms
bin_size = 0.01  # 10ms bins

ref_onsets = scope_offs_dict[(reference_struct, cond)]["start_time"].values
target_structs = [s for s in structure_labels if s != reference_struct]

with pp.destination("default"):
    fig, axes = plt.subplots(
        len(target_structs), 1, figsize=(10, 2.5 * len(target_structs)),
        sharex=True, squeeze=False,
    )
    for i, target_struct in enumerate(target_structs):
        ax = axes[i, 0]
        target_onsets = scope_offs_dict[(target_struct, cond)]["start_time"].values
        bin_centers, counts = compute_event_peth(
            ref_onsets, target_onsets, window=window, bin_size=bin_size,
        )
        rate = counts / (len(ref_onsets) * bin_size)
        ax.bar(bin_centers * 1000, rate, width=bin_size * 1000 * 0.9, color="steelblue", alpha=0.8)
        ax.axvline(0, color="k", ls="--", lw=0.8)
        ax.set_ylabel(f"{target_struct}\n(events/ref/s)")

    axes[-1, 0].set_xlabel("Time from reference OFF onset (ms)")
    fig.suptitle(
        f"{subject} | {cond} | OFF onset PETH\n"
        f"Reference: {reference_struct} (n={len(ref_onsets)} OFFs)", fontsize=12,
    )
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"off_onset_peth_{scope_label}.svg")

In [ ]:
# Full pairwise cross-correlogram matrix.
cond = scope_cond

with pp.destination("default"):
    fig, axes = plt.subplots(
        n_structures, n_structures, figsize=(2.5 * n_structures, 2 * n_structures),
        sharex=True, sharey=False,
    )
    for i, ref_struct in enumerate(structure_labels):
        ref_onsets = scope_offs_dict[(ref_struct, cond)]["start_time"].values
        for j, tgt_struct in enumerate(structure_labels):
            ax = axes[i, j]
            tgt_onsets = scope_offs_dict[(tgt_struct, cond)]["start_time"].values
            if len(ref_onsets) == 0 or len(tgt_onsets) == 0:
                ax.set_visible(False)
                continue
            bin_centers, counts = compute_event_peth(
                ref_onsets, tgt_onsets, window=window, bin_size=bin_size,
            )
            rate = counts / (len(ref_onsets) * bin_size)
            color = "gray" if i == j else "steelblue"
            ax.bar(bin_centers * 1000, rate, width=bin_size * 1000 * 0.9, color=color, alpha=0.7)
            ax.axvline(0, color="k", ls="--", lw=0.5)
            if j == 0:
                ax.set_ylabel(ref_struct, fontsize=8)
            if i == 0:
                ax.set_title(tgt_struct, fontsize=8)
            ax.tick_params(labelsize=6)

    fig.suptitle(f"{subject} | {cond} | OFF onset cross-correlograms", fontsize=12, y=1.01)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"cross_correlograms_{scope_label}.svg")

## OFF properties vs overlap

Are larger OFFs (higher `span_rel2max`) more likely to overlap with OFFs in other
structures? Do overlapping OFFs differ from local OFFs in their properties?


In [ ]:
# Annotate each OFF with its overlap count (for the chosen scope).
cond = scope_cond
overlap_counts = scope_overlap_counts

annotated_dfs = []
for struct in structure_labels:
    struct_offs = scope_offs_dict[(struct, cond)].copy()
    if struct_offs.empty:
        continue
    struct_offs["n_overlapping_structures"] = overlap_counts[struct]
    struct_offs["is_local"] = overlap_counts[struct] == 0
    annotated_dfs.append(struct_offs)

annotated_offs = pd.concat(annotated_dfs, ignore_index=True)
print(f"Total annotated OFFs ({scope_label}): {len(annotated_offs)}")
print(annotated_offs.groupby("structure", observed=True)["is_local"].value_counts())

In [ ]:
# Split violin plots: local vs overlapping OFFs.
properties_to_compare = ["median_duration", "span", "area"]

max_overlap = n_structures - 1
_flare_cmap = sns.color_palette("flare", as_cmap=True)
_flare_vals = np.linspace(0.1, 0.9, max_overlap + 1)
overlap_status_palette = {
    "Local": _flare_cmap(_flare_vals[0]),
    "Overlapping": _flare_cmap(_flare_vals[-1]),
}

with pp.destination("default"):
    fig, axes = plt.subplots(
        1, len(properties_to_compare), figsize=(5 * len(properties_to_compare), 4),
        squeeze=False,
    )
    for col_idx, prop in enumerate(properties_to_compare):
        ax = axes[0, col_idx]
        plot_df = annotated_offs[["structure", "is_local", prop]].dropna().copy()
        plot_df["overlap_status"] = plot_df["is_local"].map({True: "Local", False: "Overlapping"})
        if hasattr(plot_df["structure"], "cat"):
            plot_df["structure"] = plot_df["structure"].cat.remove_unused_categories()
        order = (
            plot_df["structure"].cat.categories.tolist()
            if hasattr(plot_df["structure"], "cat")
            else sorted(plot_df["structure"].unique())
        )
        sns.violinplot(
            data=plot_df, x="structure", y=prop, hue="overlap_status", split=True,
            inner="quart", cut=0, ax=ax, palette=overlap_status_palette, order=order,
        )
        if prop in ("area", "median_duration"):
            ax.set_yscale("log")
        ax.set_title(prop)
        ax.tick_params(axis="x", rotation=45)

    fig.suptitle(f"{subject} | {scope_cond} | OFF properties: Local vs Overlapping", fontsize=12)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"local_vs_overlapping_{scope_label}.svg")

In [ ]:
# Test whether BLAS OFFs are more likely to overlap than non-BLAS OFFs.
# Uses the pre-computed category column from the aggregated parquets.
for struct in structure_labels:
    struct_offs = annotated_offs[annotated_offs["structure"] == struct]
    if struct_offs.empty or "category" not in struct_offs.columns:
        continue

    is_blas = struct_offs["category"] == "BLAS"
    n_blas = is_blas.sum()
    n_non_blas = (~is_blas).sum()

    if n_blas == 0 or n_non_blas == 0:
        print(f"{struct}: skipping (BLAS n={n_blas}, non-BLAS n={n_non_blas})")
        continue

    frac_blas_overlap = (~struct_offs.loc[is_blas, "is_local"]).mean()
    frac_non_blas_overlap = (~struct_offs.loc[~is_blas, "is_local"]).mean()

    table = [
        [
            int((~struct_offs.loc[is_blas, "is_local"]).sum()),
            int(struct_offs.loc[is_blas, "is_local"].sum()),
        ],
        [
            int((~struct_offs.loc[~is_blas, "is_local"]).sum()),
            int(struct_offs.loc[~is_blas, "is_local"].sum()),
        ],
    ]
    odds_ratio, p_value = fisher_exact(table)
    print(
        f"{struct}: BLAS (n={n_blas}) overlap={frac_blas_overlap:.1%}, "
        f"non-BLAS (n={n_non_blas}) overlap={frac_non_blas_overlap:.1%}, "
        f"OR={odds_ratio:.2f}, p={p_value:.4f}"
    )

In [ ]:
# Box plots: OFF properties vs overlap degree.
plot_offs = annotated_offs.copy()
degree_values = sorted(plot_offs["n_overlapping_structures"].unique())
plot_offs["overlap_degree"] = pd.Categorical(
    plot_offs["n_overlapping_structures"].astype(str),
    categories=[str(d) for d in degree_values], ordered=True,
)
if hasattr(plot_offs["structure"], "cat"):
    plot_offs["structure"] = plot_offs["structure"].cat.remove_unused_categories()
structure_order = (
    plot_offs["structure"].cat.categories.tolist()
    if hasattr(plot_offs["structure"], "cat")
    else sorted(plot_offs["structure"].unique())
)
n_degrees = len(degree_values)
_flare_cmap = sns.color_palette("flare", as_cmap=True)
flare_palette = [_flare_cmap(v) for v in np.linspace(0.1, 0.9, n_degrees)]
degree_palette = {str(d): flare_palette[i] for i, d in enumerate(degree_values)}

with pp.destination("default"):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    ax = axes[0]
    plot_df = plot_offs.dropna(subset=["span_rel2max"])
    sns.boxplot(
        data=plot_df, x="structure", y="span_rel2max", hue="overlap_degree",
        hue_order=[str(d) for d in degree_values], order=structure_order, ax=ax,
        fliersize=0, palette=degree_palette, legend=False,
    )
    ax.set_xlabel("Structure")
    ax.set_ylabel("span_rel2max")
    ax.set_title("Spatial span vs overlap degree")
    for struct in structure_order:
        sdf = plot_offs[plot_offs["structure"] == struct].dropna(subset=["span_rel2max"])
        if sdf.empty:
            continue
        rho, p = spearmanr(sdf["n_overlapping_structures"], sdf["span_rel2max"])
        print(f"{struct}: span_rel2max vs overlap: rho={rho:.3f}, p={p:.2e}")

    ax = axes[1]
    sns.boxplot(
        data=plot_offs, x="structure", y="duration", hue="overlap_degree",
        hue_order=[str(d) for d in degree_values], order=structure_order, ax=ax,
        fliersize=0, palette=degree_palette, legend=False,
    )
    ax.set_yscale("log")
    ax.set_xlabel("Structure")
    ax.set_ylabel("Duration (s)")
    ax.set_title("Duration vs overlap degree")
    for struct in structure_order:
        sdf = plot_offs[plot_offs["structure"] == struct]
        if sdf.empty:
            continue
        rho, p = spearmanr(sdf["n_overlapping_structures"], sdf["duration"])
        print(f"{struct}: duration vs overlap: rho={rho:.3f}, p={p:.2e}")

    fig.suptitle(f"{subject} | {scope_cond} | OFF properties vs overlap", fontsize=12)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"properties_vs_overlap_{scope_label}.svg")

## Jitter control

Circularly shift OFF onset times within each structure by a random amount, preserving
inter-OFF-interval structure, and recompute overlap fractions. This gives a null
distribution for whether observed overlaps exceed chance.


In [ ]:
jitter_off_times = csx.jitter_off_times

In [ ]:
# Jitter control. The circular shift assumes per-condition 0-based times, so it
# is valid only for a single condition's per-condition OFFs -- not the
# absolute-time full-48h OFFs, and not the whole-recording scope.
_run_jitter = SINGLE_COND_SCOPE == "condition" and OFF_SOURCE != "morphological-full48h"
null_fracs = {}
observed_fracs = {}
if not _run_jitter:
    print(
        "Jitter control skipped: requires SINGLE_COND_SCOPE='condition' and a "
        f"per-condition OFF_SOURCE (got scope={SINGLE_COND_SCOPE!r}, "
        f"OFF_SOURCE={OFF_SOURCE!r})."
    )
else:
    n_shuffles = 200
    rng = np.random.default_rng(42)
    cond = scope_cond
    T = scope_T

    overlap_counts_obs = scope_overlap_counts
    for struct in structure_labels:
        counts = overlap_counts_obs[struct]
        observed_fracs[struct] = np.mean(counts > 0) if len(counts) > 0 else np.nan

    null_fracs = {struct: [] for struct in structure_labels}
    for shuf_idx in range(n_shuffles):
        jittered_dict = {}
        for struct in structure_labels:
            jittered_dict[(struct, cond)] = jitter_off_times(
                scope_offs_dict[(struct, cond)], T, rng
            )
        jittered_counts = compute_overlap_counts(jittered_dict, structure_labels, cond)
        for struct in structure_labels:
            c = jittered_counts[struct]
            if len(c) > 0:
                null_fracs[struct].append(np.mean(c > 0))

    print(f"=== {cond}: Jitter Control (n={n_shuffles} shuffles) ===")
    for struct in structure_labels:
        null_arr = np.array(null_fracs[struct])
        obs = observed_fracs[struct]
        if np.isnan(obs) or len(null_arr) == 0:
            continue
        p_value = np.mean(null_arr >= obs)
        print(
            f"  {struct}: observed={obs:.3f}, "
            f"null={null_arr.mean():.3f} +/- {null_arr.std():.3f}, p={p_value:.4f}"
        )

In [ ]:
# Plot observed vs null distributions (only when the jitter control ran).
if null_fracs and any(len(v) for v in null_fracs.values()):
    with pp.destination("default"):
        fig, axes = plt.subplots(1, n_structures, figsize=(3 * n_structures, 3), sharey=True)
        if n_structures == 1:
            axes = [axes]
        for ax, struct in zip(axes, structure_labels):
            null_arr = np.array(null_fracs[struct])
            obs = observed_fracs[struct]
            if np.isnan(obs) or len(null_arr) == 0:
                ax.set_visible(False)
                continue
            ax.hist(null_arr, bins=20, color="gray", alpha=0.6, label="Null")
            ax.axvline(obs, color="red", lw=2, label=f"Observed ({obs:.2f})")
            ax.set_title(struct, fontsize=9)
            ax.set_xlabel("Overlap fraction")
            if ax is axes[0]:
                ax.set_ylabel("Shuffle count")
            ax.legend(fontsize=6)
        fig.suptitle(f"{subject} | {scope_cond} | Jitter control", fontsize=12)
        fig.tight_layout()
        fig.savefig(OUTPUT_DIR / f"jitter_control_{scope_label}.svg")
else:
    print("Jitter plot skipped (no null distribution).")

## Condition comparison

Compare cross-structure overlap patterns between conditions, to assess whether sleep
pressure modulates global against local sleep.


In [ ]:
condition_comparison = []
for cond in conditions:
    overlap_counts = all_overlap_counts[cond]
    for struct in structure_labels:
        counts = overlap_counts[struct]
        if len(counts) == 0:
            continue
        condition_comparison.append(
            {
                "condition": cond,
                "structure": struct,
                "n_offs": len(counts),
                "frac_local": np.mean(counts == 0),
                "frac_any_overlap": np.mean(counts > 0),
                "mean_overlap_degree": np.mean(counts),
            }
        )

comparison_df = pd.DataFrame(condition_comparison)
print(comparison_df.to_string(index=False))

In [ ]:
palette = plots.get_condition_palette()

with pp.destination("default"):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    pivot = comparison_df.pivot(index="structure", columns="condition", values="frac_any_overlap")
    ordered_conditions = [c for c in const.CORE_CONDITIONS if c in pivot.columns]
    pivot = pivot.loc[structure_labels, ordered_conditions]
    pivot.plot(kind="bar", ax=ax, color=[palette[c] for c in ordered_conditions])
    ax.set_ylabel("Fraction overlapping")
    ax.set_title("OFFs overlapping with >= 1 other structure")
    ax.tick_params(axis="x", rotation=45)
    ax.legend(fontsize=7, bbox_to_anchor=(1.0, 1.0))

    ax = axes[1]
    pivot = comparison_df.pivot(index="structure", columns="condition", values="mean_overlap_degree")
    ordered_conditions = [c for c in const.CORE_CONDITIONS if c in pivot.columns]
    pivot = pivot.loc[structure_labels, ordered_conditions]
    pivot.plot(kind="bar", ax=ax, color=[palette[c] for c in ordered_conditions])
    ax.set_ylabel("Mean # overlapping structures")
    ax.set_title("Mean overlap degree")
    ax.tick_params(axis="x", rotation=45)
    ax.legend(fontsize=7, bbox_to_anchor=(1.0, 1.0))

    fig.suptitle(f"{subject} | Condition comparison", fontsize=13)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "condition_comparison.svg")